# M24 — Assign Blame with Backpropagation

**Objective:** understand backpropagation as structured credit assignment.

M19 showed that a parameter moves downhill when we subtract learning-rate
times the gradient. M23 named the forward graph. M24 asks **which stored
value is to blame for the loss**.

The useful whole is not a training loop. It is reverse accumulation of
local sensitivities through

`x → hidden_preactivation → hidden_activation → logits → probabilities → loss`

Framework autograd stays closed (M25). One small declared update is
allowed as local evidence, not as convergence.


## Working contract

Every experiment follows **predict → act → observe → explain**. **Predict before running**
each action cell and timestamp the prediction in your own evidence log.
A prediction is falsifiable: a sign, a number, a first mismatch, or which
named gradient should move.

Do not import a framework, do not write an epoch loop, and do not treat
a running notebook as competence. If a failure can be diagnosed by
comparing one local analytic gradient with a central finite difference,
stay there.

The repository does not prefill learner answers, ADR text, or competence.


In [ ]:
from pathlib import Path
import inspect
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ROOT = None
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "missions" / "M24" / "backprop_core.py").is_file():
        ROOT = candidate
        break
if ROOT is None:
    raise RuntimeError("Run from the LearningOS-AI repository or its labs directory.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from missions.M24.backprop_core import (
    ACTIVATION_W_NEGATIVE,
    ACTIVATION_W_POSITIVE,
    BRANCH_H,
    CLASS_AXIS,
    DEFAULT_ATOL,
    DEFAULT_EPSILON,
    DEFAULT_EPSILON_GRID,
    FORWARD_GRAPH_NODES,
    GRAPH_NODES,
    M23_GRAPH_NODES,
    ONE_STEP_INDEX,
    ONE_STEP_LEARNING_RATE,
    REFERENCE_B1,
    REFERENCE_B2,
    REFERENCE_HIDDEN_ACTIVATION,
    REFERENCE_HIDDEN_PREACTIVATION,
    REFERENCE_LOGITS,
    REFERENCE_W1,
    REFERENCE_W2,
    REFERENCE_X,
    REVERSE_GRAPH_NODES,
    SCALAR_T,
    SCALAR_V,
    SCALAR_W,
    SCALAR_X,
    SIGN_DELTA,
    SIGN_INDEX,
    TEACHING_ROW,
    TEACHING_TARGETS,
    activation_boundary_pair,
    arrays_close,
    backward_report,
    branch_gradients,
    branch_loss,
    build_scalar_chain_tape,
    central_finite_difference,
    check_parameter_gradients,
    finite_difference_sweep,
    first_mismatch,
    hidden_branch_contributions,
    network_finite_difference,
    network_loss_at_parameter,
    one_step_update,
    perturb_parameter_loss,
    reference_backward,
    relu_kink_parameter_entries,
    relu_local_derivative,
    restated_m19_invariant,
    reverse_accumulate,
    scalar_chain,
    scalar_chain_loss,
    scalar_one_step,
    softmax_nll,
    two_layer_backward,
)
from missions.M23.forward_core import (
    GRAPH_NODES as M23_FORWARD_NODES,
    two_layer_forward,
)

print("repository root:", ROOT)
print("M23 graph:", M23_GRAPH_NODES)
print("M24 graph:", GRAPH_NODES)
print("reverse order:", REVERSE_GRAPH_NODES)
print("softmax axis:", CLASS_AXIS, "loss: softmax_nll mean, targets:", TEACHING_TARGETS)
print("atol:", DEFAULT_ATOL, "epsilon:", DEFAULT_EPSILON)
print("M23 import matches bound nodes:", M23_FORWARD_NODES == M23_GRAPH_NODES)


## M19 / M23 boundary: keep the invariant and the graph

M19 froze `parameter - learning_rate * gradient` and checked the
one-parameter analytic derivative with a **central** finite difference.

M23 froze the named inference graph and class-axis softmax. M24 reuses
that contract through `missions.M23.forward_core`. It does not invent a
column-vector lecture or a `(n_out, n_in)` weight layout.

What this mission **opens:** local derivatives, chain rule, reverse
accumulation, branches, gradient checks, and one declared update.

What stays **deferred:**
- M25 — framework autograd and a training loop
- M26 — multi-cause deep-learning debugging


## Frozen teaching fixtures

Declare the useful whole **before** the first calculation.

| Fixture | Value |
| --- | --- |
| Forward graph | M23 `REFERENCE_*` with hidden ReLU |
| Targets | class `0` for both rows (softmax NLL, mean reduction) |
| Scalar chain | `x=2`, `w=0.5`, `v=-1.5`, `t=0.5`, `L=0.5*(y-t)^2` |
| Branch | `h=2` feeding `v1=1` and `v2=-0.5` |
| ReLU convention | `relu'(z) = 1` if `z > 0` else `0`, including `z = 0` |
| Finite difference | central stencil, default `epsilon=1e-5` |
| Dtype / check tol | `float64`, `atol=1e-6`, `rtol=1e-5` (proposed default; ADR unfilled) |
| Update | one declared step, never an epoch loop |

Primary sources: `3b1b-neural-networks`, `3b1b-calculus`, and
`karpathy-micrograd` in `data/source_registry.json`. Skip PyTorch here.


## Predict before running — M19 one-parameter invariant

Timestamp a prediction before `run-m19-invariant`.

On `xs=(1, 2)`, `ys=(3, 6)`, `w=1`:
- predict the mean squared loss
- predict the sign of `dL/dw` (the line `y=w x` is too shallow)
- predict that a central finite difference agrees
- predict the updated weight under `w - 0.1 * dL/dw`

The M19 formula is `dL/dw = (2/n) * sum(x * (w*x - y))`.


In [ ]:
m19 = restated_m19_invariant()
print(m19)
assert abs(m19["loss"] - 10.0) < 1e-12
assert abs(m19["analytic_gradient"] + 10.0) < 1e-12
assert abs(m19["finite_difference"] - m19["analytic_gradient"]) < 1e-8
assert abs(m19["updated_weight"] - 2.0) < 1e-12
print("M19 invariant restated: analytic, central difference, and descent update agree")


### Downhill is still subtract the gradient

Loss `10` at `w=1` with analytic gradient `-10` means increasing `w`
lowers loss. The update `1 - 0.1 * (-10) = 2` moves toward the true
slope `3`. M24 keeps this convention on every later parameter.


## Predict before running — draw the M23 graph and add a loss

Timestamp a prediction before `run-forward-graph`.

Predict:
- named intermediates still match M23's hand values
- row 0 hidden activation is `(0, 0)` and row 1 is `(1.0, 1.5)`
- mean softmax NLL with targets `(0, 0)` is the new terminal node
- reverse order starts at `loss`, not at `x`


In [ ]:
forward = two_layer_forward(REFERENCE_X, REFERENCE_W1, REFERENCE_B1, REFERENCE_W2, REFERENCE_B2)
print("M23 graph", M23_GRAPH_NODES)
print("M24 graph", GRAPH_NODES)
print("reverse", REVERSE_GRAPH_NODES)
print("hidden_preactivation", forward.hidden_preactivation)
print("hidden_activation", forward.hidden_activation)
print("logits", forward.logits)
print("probabilities", forward.probabilities)
loss = softmax_nll(forward.logits, TEACHING_TARGETS)
print("softmax_nll", loss)
assert arrays_close(forward.hidden_activation, REFERENCE_HIDDEN_ACTIVATION, atol=1e-12, rtol=0.0)
assert arrays_close(forward.logits, REFERENCE_LOGITS, atol=1e-12, rtol=0.0)
assert GRAPH_NODES[-1] == "loss"
assert REVERSE_GRAPH_NODES[0] == "loss"
print("trusted M23 forward graph plus a loss node")


### Names are the reverse-mode addresses

Row 0 is a dead hidden layer after ReLU. Row 1 is live. Credit assignment
will have to treat those rows differently. The loss is attached to
`probabilities` through softmax NLL; the numerically stable reverse step
will land on **logits**, not on a homemade softmax Jacobian.


## Predict before running — gradient sign from loss movement

Timestamp a prediction before `run-gradient-sign`.

Change **only** `W2[0, 0]` (hidden unit 0 → class-0 logit) by `±0.1`.
Same `X`, same other weights, same targets, same loss.

`W2[0, 0]` feeds the **target** class from a live hidden unit on row 1.
Predict whether `loss(w + 0.1)` is below or above `loss(w - 0.1)`, and
therefore whether `dL/dW2[0,0]` is negative or positive. Then compare
with reverse mode.


In [ ]:
sign = perturb_parameter_loss(
    REFERENCE_X, REFERENCE_W1, REFERENCE_B1, REFERENCE_W2, REFERENCE_B2,
    name="W2", index=SIGN_INDEX, delta=SIGN_DELTA,
)
backward = reference_backward()
analytic = float(backward.d_W2[SIGN_INDEX])
print("loss at w-delta, w, w+delta:", sign["loss_minus"], sign["loss_center"], sign["loss_plus"])
print("sign from loss movement", sign["predicted_sign"])
print("backprop dL/dW2[0,0]", analytic)
assert sign["loss_plus"] < sign["loss_minus"]
assert sign["predicted_sign"] < 0
assert analytic < 0
print("loss movement and reverse mode agree on sign")


### The system tells you the sign before the formula

Raising a weight that feeds the target logit lowered NLL, so the
gradient is negative. Reverse mode must reproduce that sign. If it does
not, the local rule is wrong — not the data.


## Predict before running — scalar chain rule

Timestamp a prediction before `run-scalar-chain`.

Graph: `z = w x + b → h = relu(z) → y = v h + c → L = 0.5 (y - t)^2`
with `x=2`, `w=0.5`, `v=-1.5`, `t=0.5`.

Predict, by hand:
- `z`, `h`, `y`, `L`
- `dL/dy`, `dL/dv`, `dL/dh`, `dL/dw`
- that a central difference on `w` matches `dL/dw`

Chain rule **multiplies** local derivatives along one path.


In [ ]:
trace = scalar_chain()
print("z, h, y, L", trace.z, trace.h, trace.y, trace.loss)
print("grads", {name: trace.grads[name] for name in ("y", "v", "h", "z", "w", "b", "c", "x")})
fd_w = central_finite_difference(lambda weight: scalar_chain_loss(w=weight), SCALAR_W)
print("central difference on w", fd_w)
assert abs(trace.loss - 2.0) < 1e-12
assert abs(trace.grads["w"] - 6.0) < 1e-12
assert abs(trace.grads["v"] + 2.0) < 1e-12
assert abs(fd_w - trace.grads["w"]) < 1e-8
print("scalar reverse mode matches the hand chain and the finite difference")


### Multiply along a path

`dL/dy = y - t = -2`, `dL/dh = (dL/dy) * v = 3`, `dL/dw = 3 * x = 6`.
Parameter, activation, and loss gradients are different objects that
happen to share a tape.


## Predict before running — branch accumulation

Timestamp a prediction before `run-branch`.

`h` feeds two heads: `y1 = v1 h` and `y2 = v2 h`, each with its own
squared loss. Topology and local functions stay fixed.

Predict:
- each path's contribution to `dL/dh`
- that the shared-node gradient **equals the sum**
- that omitting path 2 disagrees with a finite difference on `h`


In [ ]:
branch = branch_gradients()
omitted = branch_gradients(defect="omitted_branch")
fd_h = central_finite_difference(lambda hidden: branch_loss(h=hidden), BRANCH_H)
print("y1, y2, L", branch.y1, branch.y2, branch.loss)
print("path1, path2, sum", branch.contribution_path1, branch.contribution_path2, branch.d_h)
print("omitted dL/dh", omitted.d_h)
print("finite difference on h", fd_h)
assert abs(branch.d_h - (branch.contribution_path1 + branch.contribution_path2)) < 1e-12
assert abs(branch.d_h - 3.0) < 1e-12
assert abs(fd_h - branch.d_h) < 1e-8
assert abs(omitted.d_h - 2.0) < 1e-12
print("downstream contributions add at the branch")


### Add at a join

Path 1 contributes `2`, path 2 contributes `1`. Keeping only path 1
looks like a gradient and is still the wrong credit assignment. The
finite difference on the true loss is the discriminator.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
labels = ["path 1", "path 2", "sum (correct dL/dh)", "omitted path 2"]
values = [
    branch.contribution_path1,
    branch.contribution_path2,
    branch.d_h,
    omitted.d_h,
]
ax.bar(labels, values, color=["#4c78a8", "#4c78a8", "#2ca02c", "#d62728"])
ax.axhline(fd_h, color="black", linestyle="--", linewidth=1, label="finite difference on h")
ax.set_ylabel("contribution to dL/dh")
ax.set_title("Branch: downstream paths add")
ax.legend()
fig.tight_layout()
plt.show()
print("bar heights are the two path contributions, their sum, and the omitted-path bug")


## Predict before running — activation derivative across a hinge

Timestamp a prediction before `run-activation-derivative`.

Keep the scalar chain's downstream loss. Change **only** `w` so `z = w x`
crosses zero (`w = -0.25` → `z = -0.5`; `w = +0.25` → `z = +0.5`).

Predict:
- `relu'` and `dL/dw` when `z < 0`
- `relu'` and a nonzero `dL/dw` when `z > 0`
- that treating `relu'` as identity on the dead side disagrees with a
  finite difference


In [ ]:
dead, live = activation_boundary_pair()
wrong_dead = scalar_chain(w=ACTIVATION_W_NEGATIVE, defect="wrong_relu_derivative")
fd_dead = central_finite_difference(lambda weight: scalar_chain_loss(w=weight), ACTIVATION_W_NEGATIVE)
print("dead z, relu', dL/dw", dead.z, dead.relu_prime, dead.grads["w"])
print("live z, relu', dL/dw", live.z, live.relu_prime, live.grads["w"])
print("finite difference on dead w", fd_dead)
print("wrong relu' on dead w", wrong_dead.grads["w"])
assert dead.z < 0 and live.z > 0
assert dead.grads["w"] == 0.0
assert live.grads["w"] != 0.0
assert abs(fd_dead) < 1e-8
assert abs(wrong_dead.grads["w"]) > 1e-6
print("dead ReLU blocks credit; the identity-derivative bug does not")


### The hinge is a local rule, not a vibe

Same downstream loss, two pre-activations. Only the unit with `z > 0`
passes credit to `w`. A unit sitting **exactly** at `z = 0` is
non-differentiable; skip it as a finite-difference check rather than
"fixing" it.


## Predict before running — tiny dense-layer / softmax-NLL backward

Timestamp a prediction before `run-dense-backward`.

On the M23 batch with mean softmax NLL and targets `(0, 0)`:

- `dL/dlogits = (p - one_hot) / N`
- each hidden unit branches over the three classes
- row 0's ReLU is dead / at zero, so `dL/dZ1[0]` should be zeros

Predict that the class contributions to `dL/dh` for live unit 0 on row 1
sum to `d_hidden_activation[1, 0]`.


In [ ]:
backward = two_layer_backward(
    REFERENCE_X, REFERENCE_W1, REFERENCE_B1, REFERENCE_W2, REFERENCE_B2, TEACHING_TARGETS,
)
print(backward_report(backward))
print("dL/dlogits", backward.d_logits)
print("d_hidden_activation", backward.d_hidden_activation)
print("d_hidden_preactivation", backward.d_hidden_preactivation)
contrib = hidden_branch_contributions(backward.d_logits[1], REFERENCE_W2[0])
print("row1 unit0 class contributions", contrib, "sum", sum(contrib))
assert arrays_close(backward.d_hidden_preactivation[0], (0.0, 0.0), atol=1e-12, rtol=0.0)
assert abs(sum(contrib) - float(backward.d_hidden_activation[1, 0])) < 1e-12
print("parameter, activation, and loss gradients are named separately")


### Softmax plus NLL lands on logits

The Jacobian of softmax is not an extra homework problem here: paired
with mean NLL it is `p - y` (then divided by batch size). Hidden-unit
credit is still a **branch** over classes, which is why omitting a class
will show up in `dW1` rather than `dW2`.


## Predict before running — epsilon sweep and a ReLU hinge

Timestamp a prediction before `run-finite-difference`.

Invariant: same forward function and the same parameter.

Predict:
- `W2[0, 0]` (smooth) agrees near `epsilon=1e-5`
- very large epsilon shows truncation; very small epsilon shows roundoff
- `b1[0]` sits on a ReLU hinge (`z[0,0] = 0`) and the central difference
  is **not** a pass/fail of reverse mode


In [ ]:
analytic_w2 = float(backward.d_W2[0, 0])
sweep = finite_difference_sweep(
    lambda theta: network_loss_at_parameter(
        REFERENCE_X, REFERENCE_W1, REFERENCE_B1, REFERENCE_W2, REFERENCE_B2,
        TEACHING_TARGETS, "W2", (0, 0), theta,
    ),
    1.0,
    analytic_w2,
    DEFAULT_EPSILON_GRID,
    name="W2[0,0]",
)
for row in sweep:
    print(f"eps={row.epsilon:.0e} est={row.estimated:.6e} abs={row.absolute_error:.3e} ok={row.agrees}")
smooth = check_parameter_gradients(
    REFERENCE_X[TEACHING_ROW], REFERENCE_W1, REFERENCE_B1, REFERENCE_W2, REFERENCE_B2,
    targets=(0,),
)
print("example-1 mismatches", first_mismatch(smooth))
kinks = relu_kink_parameter_entries(REFERENCE_X, REFERENCE_HIDDEN_PREACTIVATION)
print("relu kinks", kinks)
hinge = network_finite_difference(
    REFERENCE_X, REFERENCE_W1, REFERENCE_B1, REFERENCE_W2, REFERENCE_B2,
    name="b1", index=(0,),
)
print("hinge FD on b1[0]", hinge, "analytic", float(backward.d_b1[0]))
assert first_mismatch(smooth) is None
assert ("b1", (0,)) in kinks
print("smooth checks pass; the z=0 hinge is a caveat, not a silent pass")


### Checks need a smooth neighborhood

Central differences probe `theta ± epsilon`. That is legal on `W2[0,0]`.
It is not a verdict on a unit sitting on the ReLU hinge. The
gradient-verification ADR must say which cases are in scope.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
eps = [row.epsilon for row in sweep]
err = [max(row.absolute_error, 1e-18) for row in sweep]
ax.loglog(eps, err, marker="o")
ax.set_xlabel("epsilon")
ax.set_ylabel("absolute error versus reverse mode")
ax.set_title("Central-difference error on W2[0,0]")
ax.axvline(DEFAULT_EPSILON, color="black", linestyle="--", linewidth=1, label="default epsilon")
ax.legend()
fig.tight_layout()
plt.show()
print("stable check region sits between truncation (large eps) and roundoff (tiny eps)")


## Predict before running — one declared update

Timestamp a prediction before `run-one-step`.

Change **only** `W2[0, 0]` with `learning_rate = 0.25` using
`parameter - learning_rate * gradient`. Nothing else updates. This is
not an epoch and not convergence.

Predict that loss after the step is lower than loss before, matching the
negative gradient sign already observed.


In [ ]:
step = one_step_update(
    REFERENCE_X, REFERENCE_W1, REFERENCE_B1, REFERENCE_W2, REFERENCE_B2,
    name="W2", index=ONE_STEP_INDEX, learning_rate=ONE_STEP_LEARNING_RATE,
)
scalar_step = scalar_one_step()
print(step)
print("scalar one step", scalar_step)
assert step["loss_after"] < step["loss_before"]
assert scalar_step["loss_after"] < scalar_step["loss_before"]
print("one downhill step on the declared parameter; not a training loop")


### Local movement is not convergence

The loss went down once. That supports the sign of the reverse-mode
gradient. It does not train V05, and it does not license an optimizer.


## Code reading — validate, store, reverse, accumulate, reset

Read `build_scalar_chain_tape`, `reverse_accumulate`, and
`two_layer_backward`. Predict before the next cell:

- stored values versus local derivatives
- reverse topological order
- why a second reverse pass without reset **doubles** `dL/dw`
- why omitted class paths poison `dH` but not `dW2`


In [ ]:
tape_src = inspect.getsource(reverse_accumulate)
backward_src = inspect.getsource(two_layer_backward)
print(tape_src)
print("---")
print(backward_src.split("Defects change")[0])
nodes = build_scalar_chain_tape()
first = reverse_accumulate(nodes, reset=True)
once = dict(first)
doubled = reverse_accumulate(nodes, grads=first, reset=False)
print("dL/dw once", once["w"], "without reset", doubled["w"])
assert abs(once["w"] - 6.0) < 1e-12
assert abs(doubled["w"] - 12.0) < 1e-12
print("reset zeros stored grads; skipping reset accumulates a second copy")


## Predict before running — Controlled failure: omitted branch contribution

Timestamp a prediction before `run-failure`.

Keep the M23 forward graph and the true loss. Run reverse mode with
`defect="omitted_branch"` on the **smooth** example (row 1).

Predict:
- `dW2` still matches finite differences (that local map did not use the dropped path)
- `dW1` / `db1` are the first mismatches
- the missing amount is the dropped class contributions into `dH`


In [ ]:
broken_reports = check_parameter_gradients(
    REFERENCE_X[TEACHING_ROW], REFERENCE_W1, REFERENCE_B1, REFERENCE_W2, REFERENCE_B2,
    targets=(0,), defect="omitted_branch",
)
broken_backward = two_layer_backward(
    REFERENCE_X[TEACHING_ROW], REFERENCE_W1, REFERENCE_B1, REFERENCE_W2, REFERENCE_B2,
    targets=(0,), defect="omitted_branch",
)
correct_backward = two_layer_backward(
    REFERENCE_X[TEACHING_ROW], REFERENCE_W1, REFERENCE_B1, REFERENCE_W2, REFERENCE_B2,
    targets=(0,), defect="none",
)
mismatch = first_mismatch(broken_reports)
print("first mismatch", mismatch)
print("broken dH", broken_backward.d_hidden_activation)
print("correct dH", correct_backward.d_hidden_activation)
w2_ok = [row for row in broken_reports if row.name.startswith("W2") or row.name.startswith("b2")]
w1_bad = [row for row in broken_reports if (row.name.startswith("W1") or row.name.startswith("b1")) and not row.agrees]
print("W2/b2 all agree?", all(row.agrees for row in w2_ok), "n", len(w2_ok))
print("W1/b1 mismatches", [row.name for row in w1_bad])
assert mismatch is not None
assert all(row.agrees for row in w2_ok)
assert w1_bad
print("finite differences isolate the omitted hidden-branch rule")


### Diagnose before repair

Symptom: reverse-mode numbers are finite and `dW2` even matches the
true-loss probe, but hidden-parameter checks fail.

Plausible hypotheses include an omitted class path into `dH`, a wrong
ReLU derivative, a flipped update sign, or a reduction-factor error on
every parameter. The discriminator is already on the table: checks that
**still match** (`W2`) versus checks that **do not** (`W1`). A global
scale bug would hit `W2` too. A ReLU bug would need a dead unit.

Do not repair this by changing weights or by opening a framework.


## Predict before running — Controlled failure: wrong activation derivative

Timestamp a prediction before `run-wrong-relu`.

Keep class-axis softmax and full branch accumulation. Set
`defect="wrong_relu_derivative"` (treat `relu'` as identity). Skip the
`z = 0` hinge entries; they are not this defect.

Predict:
- `W2` still matches
- a dead unit (`z < 0`, row 0 hidden unit 1) now shows a mismatch
- example 1 alone would **not** expose the bug (both hidden units live)


In [ ]:
wrong_reports = check_parameter_gradients(
    REFERENCE_X, REFERENCE_W1, REFERENCE_B1, REFERENCE_W2, REFERENCE_B2,
    defect="wrong_relu_derivative", skip_relu_kinks=True,
)
live_only = check_parameter_gradients(
    REFERENCE_X[TEACHING_ROW], REFERENCE_W1, REFERENCE_B1, REFERENCE_W2, REFERENCE_B2,
    targets=(0,), defect="wrong_relu_derivative",
)
failed = [row.name for row in wrong_reports if not row.agrees]
print("dead-unit mismatches", failed)
print("example-1 wrong-relu first mismatch", first_mismatch(live_only))
assert "b1(1,)" in failed
assert first_mismatch(live_only) is None
print("wrong relu' is visible only where a unit is actually dead")


### Two defects, two discriminators

Omitted branch fails hidden-parameter checks even on a fully live
example. Wrong `relu'` needs a negative pre-activation. Diagnosis uses
the graph and the finite-difference mask, not a vibe that "the grads
look off." Smallest repair restores one named local rule at a time.


## Predict before running — smallest repair

Timestamp a prediction before `run-failure-repair`.

Predict that `defect="none"` restores finite-difference agreement on the
smooth example, with the same `X`, weights, biases, and loss. Do not
change the teaching activation and do not add an optimizer.


In [ ]:
repaired = check_parameter_gradients(
    REFERENCE_X[TEACHING_ROW], REFERENCE_W1, REFERENCE_B1, REFERENCE_W2, REFERENCE_B2,
    targets=(0,), defect="none",
)
repaired_batch = check_parameter_gradients(
    REFERENCE_X, REFERENCE_W1, REFERENCE_B1, REFERENCE_W2, REFERENCE_B2,
    defect="none", skip_relu_kinks=True,
)
print("repaired example-1 mismatch", first_mismatch(repaired))
print("repaired batch (kinks skipped) mismatch", first_mismatch(repaired_batch))
assert first_mismatch(repaired) is None
assert first_mismatch(repaired_batch) is None
print("restored branch addition and relu' on the trusted M23 graph")


## Evidence contract

Submit, in your own log (not in this repository):

- timestamped **Predict before running** notes
- the restated M19 invariant
- the annotated named graph and reverse order
- the hand scalar-chain reverse
- branch contributions that add
- the ReLU boundary comparison
- the epsilon sweep plus the `z = 0` hinge caveat
- one-step local loss movement without a convergence claim
- omitted-branch / wrong-ReLU diagnosis (symptom, hypotheses, discriminator, repair)
- the code-reading trace

See `missions/M24/evidence_contract.yaml`. Do not paste filled evidence
into the committed notebook.


## No-AI gate

Close this notebook and complete `missions/M24/no_ai_gate.md` from a blank
page without AI-generated code, calculations, prose, or diagrams.

**Status:** [UNFILLED BY LEARNER]


## Unfilled ADR

Use `missions/M24/adr_prompt.md` to choose a V05 **gradient-verification**
policy: which parameters/cases are checked, epsilon, tolerance, when
checks run, and which failures block release. Do not claim global
optimality.

- **Status:** [UNFILLED BY LEARNER]
- **Date:** [UNFILLED BY LEARNER]
- **Owner:** [UNFILLED BY LEARNER]
- **Decision:** [UNFILLED BY LEARNER]

This notebook is not that ADR. Formal engineering review is required at M24.


## M19 → M23 → M24 → M25 handoff

M19 opened downhill updates. M23 reconstructed a multi-layer NumPy
forward pass with named intermediates. M24 assigned blame with reverse
mode, branch addition, ReLU local derivatives, and finite-difference
checks.

M25 may compare autograd to these numbers **only after** the scalar
tape, the branch sum, the dead-ReLU case, and the repaired local rule
are defended.

M25 still owns framework autograd and the training loop.


## Mission summary prompt

In your own words, using only numbers from this lab:

1. Why do downstream contributions add at a branch?
2. Why is `dL/dlogits = (p - y) / N` enough without a separate softmax Jacobian?
3. Why did omitted-branch still match `dW2`?
4. Why is a ReLU unit at exactly zero a bad finite-difference case?
5. What must M25 receive that a training-loop screenshot cannot provide?

Leave the answers in your evidence log, not in this file.


In [ ]:
assert arrays_close(forward.hidden_activation, REFERENCE_HIDDEN_ACTIVATION, atol=1e-12, rtol=0.0)
assert arrays_close(forward.logits, REFERENCE_LOGITS, atol=1e-12, rtol=0.0)
assert abs(trace.grads["w"] - 6.0) < 1e-12
assert abs(branch.d_h - 3.0) < 1e-12
assert first_mismatch(smooth) is None
assert mismatch is not None
assert first_mismatch(repaired) is None
assert step["loss_after"] < step["loss_before"]
print("M24 integrity checks passed")
